# BME 574 — Project 4 starter
### Five steps. Five cells for you to write. Everything else is done.

You are given a set of medical images, already packed into a matrix with the official
train / validation / test split. Your job is to train a convolutional network on it, find out
whether the convolution is actually earning its keep on **your** modality, and decide which
invariances are true of your images.

**Your job is the five cells marked `TODO`** — about thirty lines of PyTorch in total.

---

### What you hand in

1. **This notebook**, run top to bottom, all five `TODO` cells filled, every check passing.
2. **`REPORT.md`** — one page, the six questions in Step 6.
3. **`AI_USE.md`** — which AI tools you used and for what.

### Before you start

Run `BME574_L4_CNN_demo.ipynb` (the MNIST notebook from class) and then
`BME574_Project4_explained.ipynb`. The second one covers the three things this project is graded
on that the demo did not: the validation split, augmentation that is specific to a modality, and
how to read a model that generalises to the test set but not to the clinic.

### A warning about runtime

Everything here is sized for a laptop CPU: no step should take more than about two minutes. If a
cell is running for much longer than that, stop it and check `BATCH` and `EPOCHS` before you start
hunting for a bug.

---
## Step 0 · Load your bundle

Your track arrives as a single `.npz`. Until it does, everything runs on `demo.npz`, a small
synthetic stand-in with the same structure — the class is a little shape drawn at a random
position, so it rewards a CNN for the same reason your real images will. Use it to get your code
working, but never report results from it.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import f1_score, recall_score, confusion_matrix

BUNDLE = 'bundles_p4/demo.npz'    # <-- change to your track: blood / derma / organ
SEED = 0

torch.manual_seed(SEED)
plt.rcParams.update({'font.size': 9, 'figure.dpi': 120})
np.set_printoptions(precision=4, suppress=True)

# progress flags - each TODO sets its own to True once its checks pass
ok1 = ok2 = ok3 = ok4 = ok5 = False

print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())

In [ ]:
def load_bundle(path):
    z = np.load(path, allow_pickle=True)
    b = {k: z[k] for k in z.files}
    for k in ('track', 'split_kind', 'source', 'notes', 'license'):
        b[k] = str(b[k])
    b['class_names'] = [str(s) for s in b['class_names']]
    b['n_channels'] = int(b['n_channels'])
    if 'shift_name' in b:
        b['shift_name'] = str(b['shift_name'])
    return b


if not os.path.exists(BUNDLE):
    raise SystemExit('%s not found - ask for your track bundle.' % BUNDLE)

B = load_bundle(BUNDLE)
X_u8, y_np = B['X'], B['y']
train_idx, val_idx, test_idx = B['train_idx'], B['val_idx'], B['test_idx']
CLASSES = B['class_names']
K = len(CLASSES)
CH = B['n_channels']

print('track    :', B['track'])
print('images   :', X_u8.shape, X_u8.dtype, ' ->', CH, 'channel(s), 28 x 28')
print('classes  :', K)
print('split    :', B['split_kind'])
print('           %d train / %d val / %d test'
      % (len(train_idx), len(val_idx), len(test_idx)))
if 'shift_X' in B:
    print('extra    : shift set,', B['shift_X'].shape, '-', B['shift_name'])
print()
print(B['notes'])

### Look at the images before you model them

This is not a formality. Every decision you make in Step 3 — which flips and rotations are
legitimate — comes from what you see here. Spend a minute on it.

In [ ]:
rng = np.random.default_rng(0)
fig, ax = plt.subplots(K, 8, figsize=(8.5, 1.15 * K))
ax = np.atleast_2d(ax)
for c in range(K):
    pool = np.intersect1d(np.where(y_np == c)[0], train_idx)
    pick = rng.choice(pool, size=min(8, len(pool)), replace=False)
    for j in range(8):
        a = ax[c, j]
        a.set_xticks([]); a.set_yticks([])
        if j >= len(pick):
            a.axis('off'); continue
        img = X_u8[pick[j]]
        a.imshow(img[0], cmap='gray') if CH == 1 else a.imshow(img.transpose(1, 2, 0))
    ax[c, 0].set_ylabel(CLASSES[c][:16], fontsize=7, rotation=0, ha='right', va='center')
fig.suptitle('%s - eight training examples per class' % B['track'], y=1.005, fontsize=10)
plt.tight_layout(); plt.show()

In [ ]:
counts = np.bincount(y_np, minlength=K)
print('%-44s %8s %8s %8s %8s %8s' % ('class', 'n', 'share', 'train', 'val', 'test'))
for c in range(K):
    print('%-44s %8d %7.1f%% %8d %8d %8d'
          % (CLASSES[c][:44], counts[c], 100 * counts[c] / len(y_np),
             (y_np[train_idx] == c).sum(), (y_np[val_idx] == c).sum(),
             (y_np[test_idx] == c).sum()))

majority_class = int(np.bincount(y_np[train_idx]).argmax())
majority_acc = float((y_np[test_idx] == majority_class).mean())
print()
print('BASELINE 0 - always answer "%s": test accuracy %.4f'
      % (CLASSES[majority_class], majority_acc))

### Tensors, and the normalisation

Images arrive as `uint8` in 0–255. Two steps, and the second is the one people get wrong:

1. Scale to `[0, 1]` and cast to `float32`.
2. Standardise with the **training** mean and standard deviation — one per channel. Those two
   numbers are model parameters. Computing them over the whole dataset leaks the test set, exactly
   as fitting a `StandardScaler` on everything did in Project 3.

In [ ]:
Xf = torch.tensor(X_u8, dtype=torch.float32) / 255.0          # (n, C, 28, 28)
y = torch.tensor(y_np, dtype=torch.long)

MU = Xf[train_idx].mean(dim=(0, 2, 3), keepdim=True)          # per channel, train only
SD = Xf[train_idx].std(dim=(0, 2, 3), keepdim=True)
Xf = (Xf - MU) / SD

Xtr, ytr = Xf[train_idx], y[train_idx]
Xva, yva = Xf[val_idx], y[val_idx]
Xte, yte = Xf[test_idx], y[test_idx]

print('per-channel train mean', MU.flatten().numpy(), ' sd', SD.flatten().numpy())
print('Xtr', tuple(Xtr.shape), '  Xva', tuple(Xva.shape), '  Xte', tuple(Xte.shape))
print()
print('train mean after normalising: %+.6f  <- 0 by construction' % Xtr.mean())
print('TEST  mean after normalising: %+.6f  <- not 0, and that is correct' % Xte.mean())


def check(name, cond, hint=''):
    print(f'  {"OK  " if cond else "NOT YET"}  {name}')
    if not cond and hint:
        print('          hint: %s' % hint)
    return bool(cond)

---
## Step 1 · **TODO 1** — build the network

Write a function returning an `nn.Module` with this architecture — the same one from the MNIST
notebook, with the channel count taken from the bundle so it works on colour and greyscale alike:

```
(C, 28, 28)  ->  Conv2d(C, 16, 3, padding=1) -> ReLU -> MaxPool2d(2)   ->  (16, 14, 14)
             ->  Conv2d(16, 32, 3, padding=1) -> ReLU -> MaxPool2d(2)  ->  (32,  7,  7)
             ->  Flatten -> Dropout(p_drop) -> Linear(32*7*7, K)
```

Two things the checks will catch:

- **`padding=1` matters.** Without it a 3×3 convolution shrinks 28 to 26, then pooling gives 13,
  then 11, then 5 — and `Linear(32*7*7, K)` no longer matches. You will get a shape error, which
  is the kind one you can read.
- **No softmax at the end**, for the same reason as Project 3: `nn.CrossEntropyLoss` applies
  `log_softmax` internally.

In [ ]:
def build_cnn(n_channels, n_classes, widths=(16, 32), p_drop=0.25, seed=SEED):
    torch.manual_seed(seed)
    # ---- TODO 1 ----------------------------------------------------------
    model = None
    # ----------------------------------------------------------------------
    return model


model = build_cnn(CH, K)

_probe = None
if model is not None:
    model.eval()
    with torch.no_grad():
        _probe = model(Xtr[:4])

n_par = sum(p.numel() for p in model.parameters()) if model is not None else 0
expected = ((CH * 16 * 9 + 16) + (16 * 32 * 9 + 32) + (32 * 7 * 7 * K + K))

ok1 = all([
    check('build_cnn returns an nn.Module', isinstance(model, nn.Module)),
    check('it maps a batch of images to K logits',
          _probe is not None and tuple(_probe.shape) == (4, K),
          'check padding=1 on both convolutions, and Linear(32*7*7, K) at the end'),
    check('parameter count matches the architecture', n_par == expected,
          'expected %s, got %s' % (format(expected, ','), format(n_par, ','))),
    check('the output is logits, not probabilities',
          _probe is not None and not torch.allclose(_probe.exp().sum(1),
                                                    torch.ones(4), atol=1e-3),
          'no nn.Softmax at the end - CrossEntropyLoss does it for you'),
    check('there is a convolution in there somewhere',
          model is not None and any(isinstance(m, nn.Conv2d) for m in model.modules())),
])
print()
if ok1:
    print('%s parameters.' % format(n_par, ','))
    print('For comparison, one Linear(%d, %d) layer on the raw pixels would be %s.'
          % (CH * 784, K, format(CH * 784 * K + K, ',')))

---
## Step 2 · **TODO 2** — the training loop, with the validation set doing its job

The five lines inside the batch loop are the same five as Project 3:

```python
optimiser.zero_grad()
logits = model(xb)
loss = lossfn(logits, yb)
loss.backward()
optimiser.step()
```

What is new is **what happens between epochs**, and it is written for you — read it. After every
epoch the model is scored on the **validation** set, and if it is the best validation accuracy so
far, a copy of the weights is kept. At the end, those weights are restored.

That is model selection, and the point is that it uses the validation set and never the test set.
Choosing your best epoch by test accuracy is the quiet, common way of reporting a number that will
not survive contact with new data — you have tuned a hyperparameter (how long to train) on the
data you are claiming to be held out.

In [ ]:
import copy

def evaluate(model, Xe, ye, lossfn=None, batch=1000):
    model.eval()
    with torch.no_grad():
        logits = torch.cat([model(Xe[i:i + batch]) for i in range(0, len(Xe), batch)])
    pred = logits.argmax(1)
    acc = float((pred == ye).double().mean())
    loss = float(lossfn(logits, ye)) if lossfn is not None else float('nan')
    return acc, pred, loss


def train_model(model, Xa, ya, Xv, yv, epochs=12, lr=1e-3, batch=128, weight=None,
                augment=None, seed=SEED, verbose=True):
    loader = DataLoader(TensorDataset(Xa, ya), batch_size=batch, shuffle=True,
                        generator=torch.Generator().manual_seed(seed + 100))
    optimiser = torch.optim.Adam(model.parameters(), lr=lr)
    lossfn = nn.CrossEntropyLoss(weight=weight)
    history, best_acc, best_state, best_ep = [], -1.0, None, -1

    for ep in range(epochs):
        model.train()
        for xb, yb in loader:
            if augment is not None:
                xb = augment(xb)              # <- Step 3 plugs in here
            # ---- TODO 2 ------------------------------------------------
            pass
            # ---------------------------------------------------------------

        tr_acc, _, tr_loss = evaluate(model, Xa, ya, lossfn)
        va_acc, _, va_loss = evaluate(model, Xv, yv, lossfn)
        history.append((tr_loss, va_loss, tr_acc, va_acc))

        if va_acc > best_acc:                 # model selection, on VALIDATION only
            best_acc, best_ep = va_acc, ep
            best_state = copy.deepcopy(model.state_dict())
        if verbose:
            print('   epoch %2d   train %.4f / %.4f   val %.4f / %.4f%s'
                  % (ep + 1, tr_loss, tr_acc, va_loss, va_acc,
                     '   <- best so far' if ep == best_ep else ''))

    model.load_state_dict(best_state)          # restore the best-on-validation weights
    if verbose:
        print('   restored epoch %d, validation accuracy %.4f' % (best_ep + 1, best_acc))
    return np.array(history), best_ep


model = build_cnn(CH, K)
if not ok1:
    raise SystemExit('Finish TODO 1 first.')

import time; _t0 = time.time()
history, best_ep = train_model(model, Xtr, ytr, Xva, yva)
acc_cnn, pred_cnn, _ = evaluate(model, Xte, yte)
print('\nTEST accuracy %.4f   (trained in %.0f s)' % (acc_cnn, time.time() - _t0))

ok2 = all([
    check('the loop ran for every epoch', history.shape == (12, 4)),
    check('the training loss went down', history[-1, 0] < history[0, 0],
          'if nothing moved, the five lines are still missing'),
    check('the model beat the majority-class baseline', acc_cnn > majority_acc,
          'majority baseline is %.4f' % majority_acc),
    check('the best epoch was chosen on validation, not test',
          0 <= best_ep < 12 and history[best_ep, 3] == history[:, 3].max()),
])

### Figure 1 — training curves *(written for you)*

Three things to read here: the gap between train and validation, where the validation loss turns
around, and whether the restored epoch is where you would have stopped by eye.

In [ ]:
if not ok2:
    raise SystemExit('Finish TODO 2 first - Figure 1 needs the history.')

ep = np.arange(1, len(history) + 1)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].plot(ep, history[:, 0], color='#1F3864', label='train')
ax[0].plot(ep, history[:, 1], color='#9C2B2B', label='validation')
ax[0].set_xlabel('epoch'); ax[0].set_ylabel('cross-entropy'); ax[0].legend(fontsize=8)
ax[1].plot(ep, history[:, 2], color='#1F3864', label='train')
ax[1].plot(ep, history[:, 3], color='#9C2B2B', label='validation')
ax[1].axhline(majority_acc, color='#2E6B4F', ls='--', lw=1.1)
ax[1].text(1.2, majority_acc + .01, 'majority class', color='#2E6B4F', fontsize=7)
ax[1].set_xlabel('epoch'); ax[1].set_ylabel('accuracy'); ax[1].legend(fontsize=8)
for a in ax:
    a.axvline(best_ep + 1, color='grey', ls=':', lw=1.2)
    a.grid(alpha=.3)
ax[0].set_title('loss', fontsize=9); ax[1].set_title('accuracy', fontsize=9)
fig.suptitle('dotted line = the epoch restored, chosen on validation', y=1.02, fontsize=9)
plt.tight_layout(); plt.show()

---
## Step 3 · **TODO 3** — is the convolution earning its keep here?

The MNIST notebook showed that a CNN beats a dense net on handwritten digits, and that one fixed
pixel permutation destroys the advantage. **That was digits.** Your images are not digits, and the
question has to be asked again.

Run the same two-part test on your own track:

1. A **dense baseline** with a comparable parameter count, on the same data.
2. The **same CNN, trained on pixel-permuted images** — one fixed permutation applied to every
   image, train and test alike.

The dense baseline is written for you. You write the permutation.

If the CNN beats the dense net *and* collapses under permutation, the spatial structure is real
and the architecture is exploiting it. If the CNN barely beats the dense net, or barely notices
the permutation, then whatever is driving your accuracy is not spatial — and that is a finding
worth reporting, not a failure.

In [ ]:
if not ok2:
    raise SystemExit('Finish TODO 2 first.')

def build_dense(n_in, n_classes, width, seed=SEED):
    torch.manual_seed(seed)
    return nn.Sequential(nn.Flatten(), nn.Linear(n_in, width), nn.ReLU(),
                         nn.Linear(width, n_classes))

# a dense net sized to roughly match the CNN's parameter count
_w = max(8, round((n_par - K) / (CH * 784 + 1 + K)))
dense = build_dense(CH * 784, K, _w)
print('dense 　%d -> %d -> %d   %s parameters (CNN has %s)'
      % (CH * 784, _w, K, format(sum(p.numel() for p in dense.parameters()), ','),
         format(n_par, ',')))
_, _ = train_model(dense, Xtr, ytr, Xva, yva, verbose=False)
acc_dense, _, _ = evaluate(dense, Xte, yte)
print('dense baseline TEST accuracy %.4f' % acc_dense)

In [ ]:
# ---- TODO 3 ----------------------------------------------------------
# Build ONE fixed permutation of the CH*28*28 pixel positions and apply it to
# every image. Then train a fresh CNN on the permuted training set and score it
# on the permuted test set.
#
#   perm        a torch.randperm of length CH*784, generator seeded with SEED
#   permute(Z)  flatten each image, index with perm, reshape back to Z's shape
#   acc_perm    test accuracy of a fresh build_cnn trained on the permuted data
perm = None
def permute(Z):
    return None
acc_perm = None
# ----------------------------------------------------------------------
if acc_perm is None:
    raise SystemExit('TODO 3 is not filled in yet.')

_ref = torch.randperm(CH * 784, generator=torch.Generator().manual_seed(SEED))
ok3 = all([
    check('perm is a permutation of every pixel position',
          perm is not None and len(perm) == CH * 784
          and sorted(np.asarray(perm)) == list(range(CH * 784))),
    check('permute() keeps the image shape',
          permute(Xte[:3]) is not None and permute(Xte[:3]).shape == Xte[:3].shape),
    check('permute() is a rearrangement, not a corruption',
          torch.allclose(permute(Xte[:3]).reshape(3, -1).sort(1).values,
                         Xte[:3].reshape(3, -1).sort(1).values)),
    check('the permuted CNN was actually trained', 0.0 < acc_perm <= 1.0),
])
print()
if ok3:
    print('%-34s %8s' % ('model', 'test acc'))
    print('-' * 44)
    print('%-34s %8.4f' % ('always "%s"' % CLASSES[majority_class][:18], majority_acc))
    print('%-34s %8.4f' % ('dense, matched parameters', acc_dense))
    print('%-34s %8.4f' % ('CNN', acc_cnn))
    print('%-34s %8.4f' % ('CNN on permuted pixels', acc_perm))
    print()
    print('CNN  minus dense              : %+.4f' % (acc_cnn - acc_dense))
    print('CNN  minus permuted CNN       : %+.4f' % (acc_cnn - acc_perm))
    print()
    if acc_cnn - acc_perm > 0.02 and acc_cnn - acc_dense > 0.02:
        print('VERDICT: the convolution is doing real work on this modality. The CNN beats')
        print('         the dense net, and scrambling the pixel layout takes that away.')
    elif acc_cnn - acc_dense <= 0.02:
        print('VERDICT: the CNN is NOT clearly better than a dense net of the same size here.')
        print('         Say so, and look at the sample images for why.')
    else:
        print('VERDICT: the CNN beats the dense net but barely notices the permutation.')
        print('         Whatever it is using, it is not mainly spatial layout. Investigate.')

---
## Step 4 · **TODO 4** — the invariances that are true of *your* images

Augmentation is how you tell a network that a transformation does not change the label. You show
it transformed copies during training and keep the label the same.

Which transformations qualify is **a question about the modality, not about machine learning**,
and getting it wrong does real damage:

| | |
|---|---|
| Cells on a slide | no canonical orientation — flips and rotations are all free |
| Skin lesions | no canonical orientation either, but **colour is diagnostic** — hue and saturation jitter can erase the signal |
| CT slices | a definite left and right. A horizontal flip moves the liver to the wrong side, and if two of your classes are *femur-left* and *femur-right*, you have just mislabelled your training data |

Write an augmenter for **your** track. Available to you without any extra imports:
`torch.flip(x, dims=[3])` (horizontal), `torch.flip(x, dims=[2])` (vertical),
`torch.rot90(x, k, dims=[2, 3])`, and `torch.roll` for small translations.

Apply each transform to a **random subset of the batch**, not to all of it — the network needs to
see untransformed images too.

Then justify your choices in the memo. A wrong choice that you argued for carefully scores better
than a right one you copied.

In [ ]:
if not ok3:
    raise SystemExit('Finish TODO 3 first.')

def make_augmenter(seed=SEED):
    g = torch.Generator().manual_seed(seed + 7)
    def augment(xb):
        # ---- TODO 4 ------------------------------------------------------
        # Return an augmented copy of the batch xb, shape (n, CH, 28, 28).
        # Use torch.rand(n, generator=g) < p to pick which images to transform.
        # Choose the transforms that leave YOUR track's label unchanged, and be
        # ready to defend the ones you left out.
        return xb
        # ------------------------------------------------------------------
    return augment


augment = make_augmenter()
_batch = Xtr[:64].clone()
_aug = augment(_batch)

ok4 = all([
    check('make_augmenter returns something callable', callable(augment)),
    check('the augmenter keeps the batch shape',
          _aug is not None and _aug.shape == _batch.shape),
    check('it did NOT modify the input batch in place',
          torch.allclose(_batch, Xtr[:64]),
          'work on a copy - xb.clone() - or you corrupt the training set'),
    check('it actually changes some images',
          not torch.allclose(_aug, _batch),
          'an augmenter that returns xb unchanged is allowed only if you argue in '
          'the memo that NO transform is safe for your modality - and then say so here'),
    check('it leaves some images alone',
          not bool((_aug != _batch).reshape(len(_batch), -1).any(1).all()),
          'apply each transform to a random subset, not the whole batch'),
])

In [ ]:
if not ok4:
    raise SystemExit('Finish TODO 4 first - the figure needs the augmenter.')

_show = Xtr[:8]
_a = augment(_show.clone())
def _back(t):
    return (t * SD + MU).clamp(0, 1)
fig, ax = plt.subplots(2, 8, figsize=(8.5, 2.5))
for j in range(8):
    for r, src in enumerate((_show, _a)):
        img = _back(src)[j]
        ax[r, j].imshow(img[0], cmap='gray') if CH == 1 else ax[r, j].imshow(img.permute(1, 2, 0))
        ax[r, j].set_xticks([]); ax[r, j].set_yticks([])
ax[0, 0].set_ylabel('original', fontsize=7.5, rotation=0, ha='right', va='center')
ax[1, 0].set_ylabel('augmented', fontsize=7.5, rotation=0, ha='right', va='center')
fig.suptitle('Figure 2 - your augmentation, applied to eight training images', y=1.03, fontsize=9)
plt.tight_layout(); plt.show()
print('Look hard at the bottom row. If any of those images could not occur in a real')
print('scan of this modality, or has had its label changed by the transform, fix it now.')

---
## Step 5 · **TODO 5** — does it survive outside the test set?

Train one more CNN, this time **with your augmenter**, and compare it to the unaugmented model on
two things:

1. the ordinary **test set**, and
2. a **harder evaluation** that the test set does not cover.

The second one is the point. A held-out test set drawn from the same collection as the training
set answers "does this generalise to more images like these?" — which is not the question a
clinic is asking. Your bundle lets you ask a harder one:

- **organ track**: the bundle carries `shift_X` / `shift_y` — the same eleven organs imaged in
  the **coronal and sagittal planes**. Your model has only ever seen axial slices.
- **blood and derma tracks**: no second view exists, so the harder evaluation is the **re-oriented
  test set** — every image randomly flipped and rotated by a multiple of 90°. If you claimed in
  Step 4 that orientation does not change the label, this is where that claim is tested: the
  specimens are identical, only the slide has been turned.

The cell picks whichever applies to your track. You write the training.

In [ ]:
if not ok4:
    raise SystemExit('Finish TODO 4 first.')

# ---- TODO 5 ----------------------------------------------------------
# Train a fresh CNN with augmentation switched on, then score it.
#   model_aug   a fresh build_cnn(CH, K, seed=SEED)
#   hist_aug    train_model(..., augment=augment, verbose=False) -> the history
#   acc_aug     its accuracy on the ORDINARY test set
model_aug = None
hist_aug = None
acc_aug = None
# ----------------------------------------------------------------------
if model_aug is None or acc_aug is None:
    raise SystemExit('TODO 5 is not filled in yet.')


def reorient(Z, seed=1234):
    # Randomly flip and rotate every image by a multiple of 90 degrees.
    # For a modality with no canonical orientation this is a legitimate second
    # look at the same specimens - the slide turned on the stage. Whether that
    # is true of YOUR images is exactly the judgement you made in Step 4, and
    # this is where that judgement gets tested.
    g = torch.Generator().manual_seed(seed)
    out = Z.clone()
    k = torch.randint(0, 4, (len(Z),), generator=g)
    flip = torch.rand(len(Z), generator=g) < 0.5
    for j in range(1, 4):
        sel = k == j
        if sel.any():
            out[sel] = torch.rot90(out[sel], j, dims=[2, 3])
    out[flip] = torch.flip(out[flip], dims=[3])
    return out


if 'shift_X' in B:
    HARD = 'held-out view: %s' % B['shift_name']
    Xh = ((torch.tensor(B['shift_X'], dtype=torch.float32) / 255.0) - MU) / SD
    yh = torch.tensor(B['shift_y'], dtype=torch.long)
else:
    HARD = 'test set re-oriented (random flips and 90 degree rotations)'
    Xh, yh = reorient(Xte), yte

acc_plain_hard, _, _ = evaluate(model, Xh, yh)
acc_aug_hard, _, _ = evaluate(model_aug, Xh, yh)

ok5 = all([
    check('a second model was trained', isinstance(model_aug, nn.Module)),
    check('it is a different model from the first',
          not torch.allclose(next(model_aug.parameters()).detach(),
                             next(model.parameters()).detach()),
          'build a FRESH model - do not keep training the one from Step 2'),
    check('it was trained with the augmenter', hist_aug is not None
          and np.asarray(hist_aug).shape[1] == 4),
    check('it still works on the ordinary test set', acc_aug > majority_acc),
])
print()
if ok5:
    print('%-28s %14s %18s' % ('', 'ordinary test', 'harder evaluation'))
    print('%-28s %14s %18s' % ('', '', HARD[:18]))
    print('-' * 64)
    print('%-28s %14.4f %18.4f' % ('CNN, no augmentation', acc_cnn, acc_plain_hard))
    print('%-28s %14.4f %18.4f' % ('CNN, your augmentation', acc_aug, acc_aug_hard))
    print('-' * 64)
    print('%-28s %+14.4f %+18.4f' % ('augmentation bought', acc_aug - acc_cnn,
                                     acc_aug_hard - acc_plain_hard))

In [ ]:
if not ok5:
    raise SystemExit('Finish TODO 5 first - Figure 3 needs both models.')

labels = ['majority', 'dense', 'CNN', 'CNN\npermuted', 'CNN\n+augment']
vals = [majority_acc, acc_dense, acc_cnn, acc_perm, acc_aug]
hard = [np.nan, np.nan, acc_plain_hard, np.nan, acc_aug_hard]

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
bars = ax[0].bar(labels, vals, color=['#9A9AA5', '#9A9AA5', '#1F3864', '#9C2B2B', '#2E6B4F'],
                 width=.6)
for b, v in zip(bars, vals):
    ax[0].text(b.get_x() + b.get_width() / 2, v + .012, '%.3f' % v, ha='center', fontsize=8)
ax[0].set_ylabel('test accuracy'); ax[0].set_ylim(0, 1.1)
ax[0].set_title('Figure 3a - the whole comparison, ordinary test set', fontsize=9)
ax[0].tick_params(axis='x', labelsize=7.5)

w = 0.35
xs = np.arange(2)
ax[1].bar(xs - w / 2, [acc_cnn, acc_plain_hard], w, color='#1F3864', label='no augmentation')
ax[1].bar(xs + w / 2, [acc_aug, acc_aug_hard], w, color='#2E6B4F', label='augmented')
for i, (a_, b_) in enumerate(((acc_cnn, acc_aug), (acc_plain_hard, acc_aug_hard))):
    ax[1].text(i - w / 2, a_ + .012, '%.3f' % a_, ha='center', fontsize=8)
    ax[1].text(i + w / 2, b_ + .012, '%.3f' % b_, ha='center', fontsize=8)
ax[1].set_xticks(xs); ax[1].set_xticklabels(['ordinary\ntest set', 'harder\nevaluation'],
                                            fontsize=8)
ax[1].set_ylabel('accuracy'); ax[1].set_ylim(0, 1.1); ax[1].legend(fontsize=8)
ax[1].set_title('Figure 3b - what augmentation bought, and where', fontsize=9)
plt.tight_layout(); plt.show()

### Figure 4 — where the errors go *(written for you)*

In [ ]:
if not ok5:
    raise SystemExit('Finish TODO 5 first.')

_, pred_best, _ = evaluate(model_aug if acc_aug >= acc_cnn else model, Xte, yte)
cm = confusion_matrix(yte.numpy(), pred_best.numpy(), labels=list(range(K)))
cmn = cm / cm.sum(1, keepdims=True).clip(1)

print('%-44s %8s' % ('class', 'recall'))
for c in range(K):
    print('%-44s %8.3f' % (CLASSES[c][:44], cmn[c, c]))
print()
print('accuracy %.4f    macro-F1 %.4f'
      % (float((pred_best == yte).double().mean()),
         f1_score(yte.numpy(), pred_best.numpy(), average='macro')))

fig, ax = plt.subplots(figsize=(1.6 + 0.62 * K, 1.4 + 0.55 * K))
ax.imshow(cmn, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(K)); ax.set_xticklabels([c[:12] for c in CLASSES], rotation=40,
                                            ha='right', fontsize=7)
ax.set_yticks(range(K)); ax.set_yticklabels([c[:12] for c in CLASSES], fontsize=7)
ax.set_xlabel('predicted'); ax.set_ylabel('true')
for i in range(K):
    for j in range(K):
        if cm[i, j]:
            ax.text(j, i, '%.2f' % cmn[i, j], ha='center', va='center', fontsize=6.5,
                    color='white' if cmn[i, j] > 0.55 else '#14264A')
ax.set_title('Figure 4 - row-normalised confusion matrix', fontsize=9)
plt.tight_layout(); plt.show()

---
## Step 6 · The memo

Write **`REPORT.md`**, one page, answering these six. Short answers, real numbers, no padding.

1. **What is your track, and what does it look like?** Name the modality, the image size and
   channels, the classes and their balance, and the majority-class baseline. One sentence on what
   you noticed in the sample grid.

2. **Is the convolution earning its keep?** Give the CNN, the matched-parameter dense net, and the
   permuted CNN. Interpret the three together: what does the permutation result tell you that the
   dense comparison alone does not?

3. **Which invariances did you build in, and which did you deliberately leave out?** List the
   transforms in your augmenter and name one you considered and rejected, with the reason. This is
   a question about your imaging modality — answer it as one.

4. **What did augmentation buy on the ordinary test set, and on the harder evaluation?** Give all
   four numbers. If the gain on the test set was small and the gain on the harder evaluation was
   large, say what that means about what a test set measures.

5. **Where do the errors go?** Use the confusion matrix and the per-class recalls. Which pair of
   classes does the model confuse, and does that confusion make sense given what the images look
   like?

6. **What would you do next, with a week and the same data?** One concrete thing and what you
   expect it to change. "A deeper network" is not an answer unless you say what you think is
   currently limiting the model.

In [ ]:
# final check - everything above should be OK before you submit
steps = {'Step 1  build the CNN': ok1,
         'Step 2  training loop with validation selection': ok2,
         'Step 3  dense baseline and the permutation test': ok3,
         'Step 4  augmentation for your modality': ok4,
         'Step 5  augmented model and the harder evaluation': ok5}
for k, v in steps.items():
    print('  %s  %s' % ('OK  ' if v else 'NOT YET', k))
print()
print('All steps pass. Write REPORT.md and AI_USE.md.' if all(steps.values())
      else 'Finish the steps marked NOT YET.')

---
## If you get stuck

| Symptom | Almost always |
|---|---|
| `Expected 3-dimensional input ... but got 4-dimensional` | you passed images to a `Linear` layer without `nn.Flatten()` first |
| `mat1 and mat2 shapes cannot be multiplied (Nx800 and 1568x...)` | `padding=1` is missing from a convolution, so 28 became 26 and the flattened size is wrong |
| `Given groups=1, weight of size [16,1,3,3], expected input[...,3,28,28]` | the first `Conv2d` is hardcoded to 1 channel; use `n_channels` |
| Accuracy stuck at the majority-class share | the five lines of TODO 2 are missing, or `zero_grad()` is absent |
| The augmented model is *worse* everywhere | a transform is changing the label — a horizontal flip on CT is the usual culprit |
| `acc_perm` close to `acc_cnn` | check that you permuted **both** the training and the test images, with the *same* permutation |
| Training takes many minutes | you are on the organ track, which is the biggest; drop `epochs` to 8 while debugging |